In [ ]:
import sys
import subprocess as sp
from io import StringIO
import pandas as pd

In [2]:
def fetch_data(file_name):
    try:
        swh_query = open(file_name).read()
        data = pd.read_csv(
            StringIO(
                sp.check_output(
                    f"""pharos sql run --sql "{swh_query}" | jq -r '.result.data'""",
                    shell=True,
                ).decode("utf-8")
            )
        )
    except Exception as e:
        subject = "Pharos Query for Scopes Failed"
        body = f"{subject} on {file_name} \nError: {e}"
        print(body)
        sys.exit()
    return data

In [3]:
data_list = {}
sql_files = ["testing.sql"]

for sql_file in sql_files:
    data_list[sql_file] = fetch_data(sql_file)

2026-01-13 13:37:38,839 INFO executing SQL

WITH Parameters AS(
        SELECT 6 AS lookback_months,
            DATE_TRUNC('month', DATE_ADD('month', -6, CURRENT_DATE)) AS period_start_date,
            DATE_TRUNC('month', CURRENT_DATE) AS current_month_start
    ),
    FirstMigrationEver AS(
        SELECT cat.tenant_prefix,
            sad.enterprise_size_group,
            MIN(DATE_TRUNC('month', DATE(mel.wd_event_date))) AS first_migration_month
        FROM swh.migration_event_log AS mel
        CROSS JOIN Parameters p
        INNER JOIN swh.scopes_input_type_metrics AS sitm 
            ON mel.source_object_id = sitm.scope_external_id
            AND sitm.wd_event_date IS NOT NULL
        JOIN lookup_db.sfdc_customer_account_tenants cat 
            ON mel.source_tenant = cat.tenant_name
        JOIN lookup_db.sfdc_account_details sad 
            ON cat.sf_account_id = sad.sf_account_id
        WHERE mel.user_type = 'Customer'
            AND mel.wd_event_date IS NOT NULL
     

In [4]:
data_list["testing.sql"]

,migration_month,cumulative_me_count,cumulative_le_count
0,2025-12-01,13,31
1,2025-11-01,10,22
2,2025-10-01,4,18
3,2025-09-01,3,10
4,2025-08-01,1,5
5,2025-07-01,0,4


In [ ]:
for sql_file in sql_files:
    data_list[sql_file].to_csv(f"{sql_file.replace('.sql', '')}.csv", index=False)